# Trabajo Práctico: Calibración y Pose de la Cámara

**Universidad de Buenos Aires**  
Facultad de Ciencias Exactas y Naturales

**Materia:** Álgebra Lineal Computacional  
**Período:** Segundo cuatrimestre de 2026


## Grupo: milanesa SVD

### Integrantes

| Nombre y apellido | LU |
| --- | --- |
| Rocco Gasparini | 102/24 |
| Benjamin Pondal | 551/24 |
| Ivan Schuster | 401/24 |
| Lautaro Hildt | 1166/24 |

## Resolución


### Ejercicio 1 - DLT ecuaciones formales

**a)** Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

**1. Rango de la matriz $Q$:**

La matriz $Q$ tiene dimensiones $2n \times 12$ y el vector $m$ pertenece a $\mathbb{R}^{12}$. Por el Teorema de la Dimensión, sabemos que:
$$ \dim(\mathbb{R}^{12}) = \text{Rango}(Q) + \dim(\text{Nu}(Q)) $$
$$ 12 = \text{Rango}(Q) + \dim(\text{Nu}(Q)) $$

Como el enunciado exige una solución no nula y única a menos de un factor de escala, el núcleo de la matriz tiene que ser igual a $1$. Despejando, obtenemos que el rango necesario de la matriz es:
$$ \text{Rango}(Q) = 11 $$

**2. Cantidad mínima de puntos:**

Cada correspondencia de puntos aporta exactamente 2 ecuaciones al sistema (dos filas a la matriz $Q$). Para alcanzar un rango de 11, necesitamos como mínimo 11 filas linealmente independientes.

* Con 6 puntos obtenemos 12 filas, lo cual es suficiente para alcanzar el rango 11 requerido.

**3. Condiciones de los puntos:**
Para garantizar que el sistema alcance el rango 11, las coordenadas de los puntos elegidos no pueden ser arbitrarias.

* **Condición en el espacio 3D:**
Sin pérdida de generalidad, asumamos que todos los puntos de calibración pertenecen al plano $W = 0$. En la construcción de la matriz $Q$, las tres incógnitas de la cámara que multiplican a $W$ ($m_{13}, m_{23}$ y $m_{33}$) se multiplicarán por cero en todas las ecuaciones. Al no participar del sistema, estas 3 incógnitas quedan como variables libres, dejando a la matriz con un rango máximo de 8.

Para restringir estas 3 variables libres y lograr solución única, necesitamos aportar ecuaciones donde $W \neq 0$. Cada punto no coplanar aporta 2 ecuaciones. Si agregamos 1 solo punto fuera del plano, aportamos 2 ecuaciones, pero nos sigue quedando 1 variable libre (rango 10). Por lo tanto, es obligatorio agregar al menos 2 puntos fuera del plano base para lograr restringir el sistema. En conclusión, el conjunto total de puntos 3D no puede ser coplanar.

* **Condición en el plano de la imagen:**
Si todos los puntos proyectados en la imagen fueran colineales, podemos (sin pérdida de generalidad) rotar el sistema de referencia 2D para que caigan sobre el eje horizontal, donde $v_i = 0$ para todos los puntos. Al reemplazar $v = 0$, la ecuación impar ($f_{2i+1}$) se reduce a:
$$ U \cdot m_{21} + V \cdot m_{22} + W \cdot m_{23} + 1 \cdot m_{24} = 0 $$

Geométricamente, esta es la ecuación de un plano en el espacio 3D. 
Si todos nuestros puntos 3D satisfacen esta ecuación, significa que viven sobre ese único plano. Sin embargo, por la condición anterior ya demostramos que el conjunto de puntos 3D no puede ser coplanar. 
Por lo tanto, la única forma de que un conjunto no coplanar de puntos satisfaga esa ecuación simultáneamente es que los coeficientes del plano sean nulos: **$m_{21} = m_{22} = m_{23} = m_{24} = 0$**. 

Al forzar a cero estas 4 variables, nos quedan 8 incógnitas por despejar ($m_1$ y $m_3$). Como las ecuaciones impares ya se consumieron forzando los ceros, solo nos quedan las filas pares (las de $u$). Utilizando el mínimo de 6 puntos, disponemos de apenas 6 ecuaciones útiles para resolver 8 incógnitas. 
Este sistema subdeterminado garantiza la aparición de al menos 2 variables libres adicionales, provocando infinitas soluciones. Por ende, para que el algoritmo funcione, es estrictamente necesario que los puntos proyectados en la imagen no sean todos colineales.

**b)** Encontrar el vector $m$ que resuelve la minimización de la ecuación $8$.

El objetivo es encontrar el vector $m$ que minimice la forma cuadrática $m^T Q^T Q m$, sujeto a la restricción $\|m\| = 1$. Para resolverlo, utilizamos la Descomposición en Valores Singulares (SVD) de la matriz $Q = U \Sigma V^T$. Reemplazando $Q$ en la función a minimizar, obtenemos $m^T V (\Sigma^T \Sigma) V^T m$.

Para simplificar la expresión, definimos el vector $y = V^T m$ y calculamos su norma al cuadrado para verificar cómo se ve afectada la restricción inicial:
$$ \|y\|^2 = y^T y = (V^T m)^T (V^T m) = m^T V V^T m $$

Dado que $V$ es una matriz ortogonal proveniente de la SVD, se cumple que $V V^T = I$. Por lo tanto:
$$ \|y\|^2 = m^T I m = m^T m = \|m\|^2 $$

Como la norma inicial requerida era $\|m\| = 1$, queda demostrado que la norma del nuevo vector $y$ también es estrictamente 1.

Sabiendo que $y = V^T m$ y $y^T = m^T V$, el problema se transforma en minimizar la expresión $y^T (\Sigma^T \Sigma) y$ sujeta a la restricción $\|y\|^2 = 1$. La matriz $\Sigma^T \Sigma$ es diagonal y contiene los valores singulares al cuadrado ordenados de mayor a menor ($\sigma_1^2 \ge \sigma_2^2 \ge \dots \ge \sigma_{12}^2 \ge 0$). Al desarrollar este producto matricial, obtenemos una sumatoria:
$$ y^T (\Sigma^T \Sigma) y = \sum_{i=1}^{12} \sigma_i^2 y_i^2 $$

Como los valores singulares están ordenados, cualquier $\sigma_i^2$ es mayor o igual al último valor $\sigma_{12}^2$. Esto nos permite armar la siguiente desigualdad para acotar la función por debajo:
$$ \sum_{i=1}^{12} \sigma_i^2 y_i^2 \ge \sum_{i=1}^{12} \sigma_{12}^2 y_i^2 $$

Sacando factor común el término constante en el lado derecho:
$$ \sum_{i=1}^{12} \sigma_i^2 y_i^2 \ge \sigma_{12}^2 \left( \sum_{i=1}^{12} y_i^2 \right) $$

Por la restricción comprobada anteriormente, sabemos que la sumatoria de los $y_i^2$ es exactamente igual a la norma al cuadrado de $y$, que es 1. Sustituyendo esto, llegamos a la cota:
$$ \sum_{i=1}^{12} \sigma_i^2 y_i^2 \ge \sigma_{12}^2 $$

Para que un valor sea efectivamente el mínimo de una función, basta con verificar que la función es siempre mayor o igual a ese valor, y además demostrar que existe al menos un vector que hace posible alcanzar exactamente ese valor. Para que nuestra sumatoria sea igual a $\sigma_{12}^2$, basta con proponer un vector $y$ que asigne todo el peso a la última variable y anule las demás:
$$ y = \begin{bmatrix} 0 \\ 0 \\ \vdots \\ 1 \end{bmatrix} $$

Este vector propuesto cumple con la restricción de norma unitaria y hace que la función tome el valor exacto $\sigma_{12}^2$, demostrando de forma rigurosa que dicho valor es el mínimo buscado.

Finalmente, a partir de la definición $y = V^T m$, multiplicamos a izquierda por $V$ de ambos lados para despejar la variable original:
$$ V y = V V^T m \implies m = V y $$

Al multiplicar la matriz $V$ por el vector $y$ propuesto, el resultado extrae la última columna de la matriz. En conclusión, el vector $m$ de norma unitaria que resuelve la minimización original corresponde a la última columna de la matriz ortogonal $V$, la cual se obtiene al aplicar la SVD sobre la matriz de proyecciones $Q$.

### Ejercicio 2 - Calibración con puntos 3D

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

Calcular un nuevo set de parámetros para 

b) $\alpha_u = \alpha_v$ 

c) $K_{12}=0$



### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


#### Ítem 1 - Condición sobre el rango de $Q$

¿Qué rango debe tener $Q\in\mathbb{R}^{2n\times9}$ para asegurar una solución única de $H$ a menos de un factor de escala?

La matriz $Q\in\mathbb{R}^{2n\times 9}$ debe tener rango 8 para asegurar una solución única de $H$ a menos de un factor de escala.

El vector $h$ contiene los nueve elementos de la matriz $H$ y pertenece a $\mathbb{R}^{9}$. El conjunto de soluciones de la ecuación $Qh=0$ es el núcleo de $Q$. El vector $h=0$ siempre es solución. Sin embargo, produciría $H=0$, que no representa una homografía válida. Por eso buscamos una solución no nula.

Por otro lado, buscamos una solución única de $H$ a menos de un factor de escala. Esto quiere decir que, si $H_1$ y $H_2$ son dos soluciones no nulas, entonces $H_1=cH_2$ para algún escalar $c\neq0$. Multiplicar $H$ por un escalar no nulo no cambia las coordenadas de la imagen, porque ese factor se cancela al dividir por la tercera coordenada homogénea. Para que todas las soluciones no nulas sean múltiplos entre sí, el espacio de soluciones debe tener dimensión uno o, lo que es lo mismo, $\dim(\operatorname{Nu} Q)=1$.

Como $Q$ tiene nueve columnas, representa una transformación lineal con dominio $\mathbb{R}^{9}$. Por el teorema de la dimensión,

$$
\dim(\operatorname{Im} Q)+\dim(\operatorname{Nu} Q)=9.
$$

Recordemos que $\dim(\operatorname{Im} Q)=\operatorname{rg}(Q)$ y acabamos de ver que $\dim(\operatorname{Nu} Q)=1$. Luego,

$$
\operatorname{rg}(Q)+1=9,
$$

por lo tanto, $\operatorname{rg}(Q)=8$.


#### Ítem 2 - Construcción de $Q$

`obtenerQ2d(imgPts, objPts)` recibe las correspondencias entre puntos del tablero $(U_i,V_i)$ y de la imagen $(u_i,v_i)$, en el mismo orden. Cada correspondencia aporta las dos ecuaciones de la ecuación 11. Al apilarlas se obtiene $Q\in\mathbb{R}^{2n\times9}$ para el sistema $Qh=0$, donde $h$ contiene las entradas de $H$ ordenadas por filas.

Para obtener las homografías que se usan en el ítem siguiente, normalizamos los puntos según la sección 2.3, construimos $Q$ con esos puntos normalizados y elegimos el autovector del menor autovalor de $Q^TQ$, pidiendo que $\|h\|_2=1$. Por último, desnormalizamos $H$ para que vuelva a relacionar las coordenadas originales del tablero y de la imagen: $H=T_{\mathrm{img}}^{-1}\check H T_{\mathrm{obj}}$.


#### Ítem 3 - Construcción de $A$

`armarA(Hlst)` recibe una lista de homografías, cada una obtenida de una imagen distinta. Como la matriz de rotación es ortogonal entonces, en particular, las dos primeras columnas de la rotación son ortogonales entre si, y ambas tienen norma 1. Esto nos provee de dos restricciones por homografía: $c_1^TBc_2=0$ y $c_1^TBc_1-c_2^TBc_2=0$, con $B=K^{-T}K^{-1}$ y $c_1,c_2$ columnas de $H$.

Apilamos sus coeficientes como filas $v_{12}^T$ y $(v_{11}-v_{22})^T$, en el orden de incógnitas $b=(B_{11},B_{12},B_{22},B_{13},B_{23},B_{33})^T$. Así, obtenemos $Ab=0$, con $A\in\mathbb{R}^{2N\times6}$ para $N$ homografías.


a) Obtener la matriz $\mathbf{K}$

b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.

### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---